# Preuve — SciFact, recherche documentaire ancrée

Vérifier une affirmation scientifique en retrouvant la preuve dans la
littérature. 5 183 documents, 300 requêtes de test, métrique nDCG@10.

## Repères publiés

| Système | nDCG@10 | Source |
|---|---|---|
| BM25 | 0,665 | BEIR, Thakur et al. 2021 |
| BM25 + cross-encodeur MiniLM | 0,688 | BEIR, meilleur du papier |
| E5-large-v2 | 0,7224 | fiche MTEB |
| Snowflake arctic-embed-l | 0,7382 | fiche MTEB |
| BGE-large-en-v1.5 | 0,7461 | fiche MTEB |
| **GTE-large-en-v1.5** | **0,8243** | fiche MTEB — **la cible** |

## Ce qui est déjà établi

Mesuré lors des exécutions précédentes, avec deux points de contrôle validés :

| Mesure | Valeur | |
|---|---|---|
| BM25 réimplémenté | 0,6756 | contrôle, +0,011 vs publié |
| BGE-large reproduit | 0,7463 | contrôle, **+0,0002** vs publié |
| hybride BM25 + BGE | 0,7583 | |
| BM25 → reclassement | 0,7310 | le reclasseur **aide** BM25 (+0,055) |
| hybride → reclassement | 0,7368 | mais **dégrade** l'hybride (−0,022) |
| plafond du reclassement top-100 | 0,9265 | |

**Leçon retenue et corrigée ici** : le cross-encodeur réordonne intégralement les
candidats, donc il écrase le classement de la fusion quand celui-ci est meilleur
que lui. On ne remplace plus l'ordre — on le **fusionne**, et on mesure les deux.

## Marche à suivre

1. Exécution → Modifier le type d'exécution → **GPU**.
2. Cellule 1, puis **Exécution → Redémarrer la session** (obligatoire).
3. Tout le reste dans l'ordre.

> Si une cellule échoue sur `CUDA error: device-side assert triggered`, le
> contexte CUDA est empoisonné pour tout le processus : redémarrer la session
> est le seul remède.

## 1. Versions — à exécuter en premier, puis redémarrer

`gte-large-en-v1.5` charge un `modeling.py` maison écrit pour transformers 4.
Sur transformers 5, il déclenche un `device-side assert`. On épingle la version
avant tout import.

In [ ]:
!pip -q install "transformers>=4.44,<5" "sentence-transformers>=3.0,<4" nltk 2>&1 | tail -3

print()
print("=" * 64)
print("  Exécution → Redémarrer la session, PUIS reprendre à la cellule 2.")
print("  Sans redémarrage, l'ancien transformers reste chargé en mémoire.")
print("=" * 64)

## 2. Vérifications

In [ ]:
import subprocess, transformers, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
      or "AUCUN GPU — Exécution → Modifier le type d'exécution → GPU")
print("transformers", transformers.__version__, "| torch", torch.__version__,
      "| cuda", torch.cuda.is_available())
assert transformers.__version__.startswith("4"), (
    f"transformers {transformers.__version__} : le pin n'a pas pris. "
    "Relance la cellule 1, puis Exécution → Redémarrer la session.")
print("→ versions conformes")

## 3. Données — téléchargées directement, rien à téléverser

In [ ]:
import json, csv, zipfile, urllib.request, os
import numpy as np

if not os.path.exists("scifact"):
    urllib.request.urlretrieve(
        "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip",
        "scifact.zip")
    zipfile.ZipFile("scifact.zip").extractall(".")

corpus = [json.loads(l) for l in open("scifact/corpus.jsonl", encoding="utf-8")]
requetes = {json.loads(l)["_id"]: json.loads(l)["text"]
            for l in open("scifact/queries.jsonl", encoding="utf-8")}
qrels = {}
with open("scifact/qrels/test.tsv") as f:
    r = csv.reader(f, delimiter="\t"); next(r)
    for qid, did, s in r:
        qrels.setdefault(qid, {})[did] = int(s)

ids = [d["_id"] for d in corpus]
textes = [f"{d['title']} {d['text']}".strip() for d in corpus]
position = {d: i for i, d in enumerate(ids)}

assert len(corpus) == 5183 and len(qrels) == 300, "les données ne correspondent pas"
print(f"{len(corpus)} documents | {len(qrels)} requêtes de test | contrôle données conforme")

## 4. Métriques — définies comme trec_eval, que BEIR utilise

In [ ]:
def ndcg(classement, pertinents, k=10):
    gains = np.array([pertinents.get(d, 0) for d in classement[:k]], dtype=np.float64)
    dcg = float((gains / np.log2(np.arange(2, len(gains) + 2))).sum())
    ideal = np.array(sorted(pertinents.values(), reverse=True)[:k], dtype=np.float64)
    if ideal.size == 0:
        return 0.0
    idcg = float((ideal / np.log2(np.arange(2, ideal.size + 2))).sum())
    return dcg / idcg if idcg > 0 else 0.0

def rappel(classement, pertinents, k=100):
    total = sum(1 for v in pertinents.values() if v > 0)
    return sum(1 for d in classement[:k] if pertinents.get(d, 0) > 0) / total if total else 0.0

MESURES = {}

def evaluer(resultats, nom, attendu=None, tolerance=0.02):
    v = float(np.mean([ndcg(resultats[q], qrels[q], 10) for q in qrels]))
    r = float(np.mean([rappel(resultats[q], qrels[q], 100) for q in qrels]))
    MESURES[nom] = v
    ligne = f"{nom:<36} nDCG@10 {v:.4f}   Recall@100 {r:.4f}"
    if attendu is not None:
        e = v - attendu
        ligne += (f"   | attendu {attendu:.4f}, écart {e:+.4f} → "
                  f"{'conforme' if abs(e) <= tolerance else 'ÉCART — à comprendre'}")
    print(ligne)
    return v

def classer(scores, k=100):
    top = np.argpartition(-scores, k)[:k]
    return [ids[i] for i in top[np.argsort(-scores[top])]]

## 5. BM25 — premier point de contrôle

Réglages d'Anserini (`k1=0.9`, `b=0.4`), racinisation Porter, liste d'arrêt de
Lucene, titre et texte concaténés. **Attendu : 0,6756.**

In [ ]:
import re
from collections import Counter
from nltk.stem.porter import PorterStemmer

ARRET = set('''a an and are as at be but by for if in into is it no not of on or
such that the their then there these they this to was will with'''.split())
_r, _cache = PorterStemmer(), {}

def normaliser(t):
    out = []
    for m in re.findall(r"[a-z0-9]+", t.lower()):
        if m in ARRET:
            continue
        s = _cache.get(m)
        if s is None:
            s = _r.stem(m); _cache[m] = s
        out.append(s)
    return out

class BM25:
    def __init__(self, docs, k1=0.9, b=0.4):
        self.k1, self.b = k1, b
        jetons = [normaliser(d) for d in docs]
        self.n = len(jetons)
        self.longueurs = np.array([len(d) for d in jetons], dtype=np.float32)
        self.moyenne = float(self.longueurs.mean())
        brut = {}
        for i, doc in enumerate(jetons):
            for t, f in Counter(doc).items():
                brut.setdefault(t, []).append((i, f))
        self.index = {}
        for t, post in brut.items():
            idx = np.array([p[0] for p in post], dtype=np.int32)
            frq = np.array([p[1] for p in post], dtype=np.float32)
            df = len(post)
            self.index[t] = (idx, frq, float(np.log(1 + (self.n - df + .5) / (df + .5))))
    def scores(self, q):
        s = np.zeros(self.n, dtype=np.float32)
        for t in normaliser(q):
            e = self.index.get(t)
            if e is None:
                continue
            idx, frq, idf = e
            norme = 1 - self.b + self.b * self.longueurs[idx] / self.moyenne
            s[idx] += idf * (frq * (self.k1 + 1)) / (frq + self.k1 * norme)
        return s

lex = BM25(textes)
scores_bm25 = {q: lex.scores(requetes[q]) for q in qrels}
res_bm25 = {q: classer(scores_bm25[q]) for q in qrels}
evaluer(res_bm25, "BM25 réimplémenté", attendu=0.6756)

## 6. GTE-large-en-v1.5 — deuxième point de contrôle

Le modèle à battre. **Attendu : 0,8243.**

Deux garde-fous, tirés d'un échec précédent : un essai à blanc sur CPU avant de
toucher au GPU, pour qu'un code maison cassé n'empoisonne pas le contexte CUDA.

In [ ]:
from sentence_transformers import SentenceTransformer

DENSE = "Alibaba-NLP/gte-large-en-v1.5"

print("essai à blanc sur CPU…")
_e = SentenceTransformer(DENSE, trust_remote_code=True, device="cpu")
_v = _e.encode(["test de chargement"], convert_to_numpy=True)
assert _v.shape[1] == 1024, f"dimension inattendue : {_v.shape}"
print(f"  dimension {_v.shape[1]} — le modèle fonctionne, passage au GPU")
del _e

dense = SentenceTransformer(DENSE, trust_remote_code=True, device="cuda")
dense.max_seq_length = 512   # 12 % du corpus seulement dépasse ce seuil

# GTE n'attend aucun préfixe, ni côté requête ni côté document.
V_docs = dense.encode(textes, batch_size=32, normalize_embeddings=True,
                      show_progress_bar=True, convert_to_numpy=True)
qids = list(qrels)
V_req = dense.encode([requetes[q] for q in qids], batch_size=32,
                     normalize_embeddings=True, convert_to_numpy=True)

scores_dense = {q: V_docs @ v for q, v in zip(qids, V_req)}
res_dense = {q: classer(scores_dense[q]) for q in qrels}
evaluer(res_dense, "GTE-large-en-v1.5", attendu=0.8243)

## 7. Fusion hybride

Fusion par rang réciproque : chaque système vote par le rang qu'il attribue, ce
qui évite de ramener un BM25 non borné et un cosinus sur une même échelle.
On balaie `K` et le poids de BM25 plutôt que de les fixer au hasard.

In [ ]:
K_RRF_DEFAUT = 60

def rangs(scores, profondeur=200):
    top = np.argpartition(-scores, profondeur)[:profondeur]
    return {int(i): r for r, i in enumerate(top[np.argsort(-scores[top])])}

def fusionner_rangs(listes, poids, K=60, k=100):
    f = {}
    for rgs, p in zip(listes, poids):
        for i, r in rgs.items():
            f[i] = f.get(i, 0.0) + p / (K + r + 1)
    return [ids[i] for i, _ in sorted(f.items(), key=lambda kv: -kv[1])[:k]]

rangs_bm25  = {q: rangs(scores_bm25[q])  for q in qrels}
rangs_dense = {q: rangs(scores_dense[q]) for q in qrels}

print(f"{'K':>5}{'poids BM25':>12}{'nDCG@10':>10}")
meilleur = (None, -1.0)
for K in (10, 20, 60):
    for p in (0.2, 0.3, 0.5, 0.7, 1.0):
        res = {q: fusionner_rangs([rangs_bm25[q], rangs_dense[q]], [p, 1.0], K=K) for q in qrels}
        v = float(np.mean([ndcg(res[q], qrels[q], 10) for q in qrels]))
        if v > meilleur[1]:
            meilleur = ((K, p), v)
        print(f"{K:>5}{p:>12.1f}{v:>10.4f}")

(K_opt, p_opt), _ = meilleur
print(f"\nmeilleure fusion : K={K_opt}, poids BM25={p_opt}")
res_hybride = {q: fusionner_rangs([rangs_bm25[q], rangs_dense[q]], [p_opt, 1.0], K=K_opt)
               for q in qrels}
evaluer(res_hybride, "hybride BM25 + GTE")

## 8. Reclassement — fusionné, pas substitué

Mesuré à l'exécution précédente : le cross-encodeur fait gagner **+0,055** à BM25
mais fait **perdre 0,022** à l'hybride. La raison est mécanique — il réordonne
intégralement, donc il écrase un classement de départ meilleur que le sien.

On calcule donc son classement, puis on rapporte **les deux variantes** :
substitution et fusion. La fusion conserve le signal du premier étage.

In [ ]:
import time
from sentence_transformers import CrossEncoder

RECLASSEUR = "BAAI/bge-reranker-v2-m3"
K_RECLASSE = 100

ce = CrossEncoder(RECLASSEUR, max_length=512, device="cuda")
# fp16 applique apres construction : `model_kwargs` n'existe qu'a partir de
# sentence-transformers 4, et on est epingle en 3.x a cause de GTE.
ce.model.half()

def scores_reclasseur(candidats, k=K_RECLASSE, nom=""):
    # Toutes les paires en un seul appel : le GPU sature mieux et la barre
    # de progression devient réelle.
    ordre = list(qrels)
    paires, bornes = [], []
    for q in ordre:
        docs = candidats[q][:k]
        bornes.append((len(paires), len(paires) + len(docs), docs))
        paires.extend((requetes[q], textes[position[d]]) for d in docs)
    print(f"\n{nom} : {len(paires)} paires")
    t0 = time.time()
    s = ce.predict(paires, batch_size=128, show_progress_bar=True)
    d = time.time() - t0
    print(f"  {d:.0f} s — {1000*d/len(ordre):.0f} ms par requête")
    return {q: list(zip(docs, s[a:b])) for q, (a, b, docs) in zip(ordre, bornes)}

notes = scores_reclasseur(res_hybride, nom="reclassement de l'hybride")

# (a) substitution : l'ordre du reclasseur remplace celui de la fusion
res_sub = {q: [d for d, _ in sorted(v, key=lambda x: -x[1])] for q, v in notes.items()}
evaluer(res_sub, "hybride → reclassement (substitution)")

# (b) fusion : les deux classements votent
def rangs_liste(liste):
    return {position[d]: r for r, d in enumerate(liste)}

print(f"\n{'K':>5}{'poids reclasseur':>18}{'nDCG@10':>10}")
meilleur_ce = (None, -1.0)
for K in (10, 20, 60):
    for p in (0.5, 1.0, 1.5, 2.0, 3.0):
        res = {q: fusionner_rangs(
                   [rangs_liste(res_hybride[q]), rangs_liste(res_sub[q])], [1.0, p], K=K)
               for q in qrels}
        v = float(np.mean([ndcg(res[q], qrels[q], 10) for q in qrels]))
        if v > meilleur_ce[1]:
            meilleur_ce = ((K, p), v)
        print(f"{K:>5}{p:>18.1f}{v:>10.4f}")

(K_ce, p_ce), _ = meilleur_ce
print(f"\nmeilleure fusion : K={K_ce}, poids reclasseur={p_ce}")
res_final = {q: fusionner_rangs(
                 [rangs_liste(res_hybride[q]), rangs_liste(res_sub[q])], [1.0, p_ce], K=K_ce)
             for q in qrels}
evaluer(res_final, "hybride + reclassement (fusion)")

## 9. Plafond — ce que ce premier étage permet au mieux

Un reclasseur parfait, qui remonterait en tête tous les documents pertinents
déjà présents dans les candidats. Ça borne ce qu'on peut espérer sans améliorer
la récupération elle-même.

In [ ]:
parfait = {}
for q in qrels:
    l = res_hybride[q]
    pert = [d for d in l if qrels[q].get(d, 0) > 0]
    parfait[q] = pert + [d for d in l if d not in set(pert)]
evaluer(parfait, "plafond du premier étage (top-100)")

## 10. Tableau final

In [ ]:
PUBLIES = [
    ("BM25 (BEIR 2021)",             0.6650, "publié"),
    ("BM25 + CE MiniLM (BEIR 2021)", 0.6880, "publié"),
    ("E5-large-v2",                  0.7224, "publié"),
    ("Snowflake arctic-embed-l",     0.7382, "publié"),
    ("BGE-large-en-v1.5",            0.7461, "publié"),
    ("GTE-large-en-v1.5",            0.8243, "publié — la cible"),
]
CIBLE = 0.8243

print(f"{'système':<44}{'nDCG@10':>10}   statut")
print("-" * 78)
for nom, v, st in PUBLIES:
    print(f"{nom:<44}{v:>10.4f}   {st}")
print("-" * 78)
for nom, v in MESURES.items():
    marque = "  ← dépasse la cible" if v > CIBLE else ""
    print(f"{nom:<44}{v:>10.4f}   mesuré{marque}")
print("-" * 78)

json.dump({"publies": {n: v for n, v, _ in PUBLIES},
           "mesures": MESURES,
           "cible": CIBLE,
           "n_requetes": len(qrels),
           "dense": DENSE, "reclasseur": RECLASSEUR},
          open("resultats_scifact.json", "w"), indent=2)
print("écrit : resultats_scifact.json")

## 11. Récupérer

In [ ]:
from google.colab import files
files.download("resultats_scifact.json")